# Chai-1 SMILES arm, structures saved — 21-entry KRAS covalent holdout

Re-runs the Chai-1 0.6.1 **no-bond-declared** arm on the 21 post-cutoff KRAS entries and, this time, keeps every predicted structure. The 20 Oct run scored on Colab and the CIFs were never pulled off, so the nucleophile-identity test (`results/2026-10-28_nucleophile-identity/`) has Boltz-2 and Protenix columns and no Chai column.

Same runner as every other arm (`kras_holdout.py`, sha256 d17a5e53…), same seed 42 / 5 samples / 200 steps / 3 recycles, ESM embeddings. 21 entries × 5 samples = 105 structures. Roughly 30–45 min on an A100.

**Persistence:** outputs go to this account's Drive at `MyDrive/kras_chai_smiles_2026-10-28/` AND download as a zip at the end. Check the Drive folder in the web UI, not just `ls` on the mount.

Run all. Closing the tab after cell 4 starts does not stop the run; cell 5 shows progress.

## 1 · GPU and Drive

In [ ]:
import torch, os, datetime
print("cuda:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/kras_chai_smiles_2026-10-28'
os.makedirs(DEST, exist_ok=True)
open(f"{DEST}/MOUNT_MARKER.txt","w").write(f"mounted {datetime.datetime.utcnow().isoformat()}Z\n")
print("DEST:", DEST, "->", os.listdir(DEST))

## 2 · Write runner + manifest (byte-identical to the rerun_2026-10-23 copies in the public repo)

In [ ]:
import base64, pathlib, os, hashlib
os.chdir('/content')
BLOBS = {
"kras_holdout.py": "...",
"run_manifest.csv": "cGRiX2lkLHRpZXIscmVzaWR1ZV9uYW1lLHJlc2lkdWVfbnVtLHJlc2lkdWVfYXRvbSxsaWdhbmRfY29tcF9pZCxsaWdhbmRfYXRvbSxkZXBvc2l0ZWRfYm9uZF9BLHJlbGVhc2VfZGF0ZSxwb3N0X2JvbHR6Mixwb3N0X3Byb3Rlbml4XzIwMjUwNjMwCjlCQUksQV9ub25DeXNfVHlyLFRZUiw3MSxPSCxXTVUsUywxLjY2OSwyMDI1LTA0LTA5LHRydWUsZmFsc2UKOUJBSixBX25vbkN5c19UeXIsVFlSLDcxLE9ILEExQUs4LFMwMiwxLjY2MSwyMDI1LTA0LTA5LHRydWUsZmFsc2UKOUJBSyxBX25vbkN5c19UeXIsVFlSLDcxLE9ILFdOMyxTLDEuNjQ1LDIwMjUtMDQtMDksdHJ1ZSxmYWxzZQo4VDRWLEFfbm9uQ3lzX0FzcCxBU1AsMTIsT0QyLFk2MyxDMjQsMS40NjYsMjAyNC0wMy0xMyx0cnVlLGZhbHNlCjlHQkosQV9ub25DeXNfQXNwLEFTUCwxMixDRyxBMUlKNyxONDEsMS42MzIsMjAyNS0wNC0zMCx0cnVlLGZhbHNlCjlRUTAsQV9ub25DeXNfQXNwLEFTUCwxMixPRDIsQTFJOUUsQzQ2LDEuNDM0LDIwMjUtMDctMjMsdHJ1ZSx0cnVlCjlITVIsQl9DeXNfcG9zdFByb3Rlbml4LENZUyw5MixTRyxBMUlXRSxDMTksMS43NzEsMjAyNS0xMi0yNCx0cnVlLHRydWUKOUk3WSxCX0N5c19wb3N0UHJvdGVuaXgsQ1lTLDEzLFNHLEExSTA4LEMyMSwxLjgxNCwyMDI1LTExLTEyLHRydWUsdHJ1ZQo5TjQ0LEJfQ3lzX3Bvc3RQcm90ZW5peCxDWVMsMTIsU0csQTFCVjcsQzcsMS43OTksMjAyNS0xMi0xMCx0cnVlLHRydWUKOU5aTSxCX0N5c19wb3N0UHJvdGVuaXgsQ1lTLDEyLFNHLEExQjdQLEMzNywxLjgzMCwyMDI1LTExLTA1LHRydWUsdHJ1ZQo5TlpOLEJfQ3lzX3Bvc3RQcm90ZW5peCxDWVMsMTIsU0csQTFCN1EsQzM2LDEuODI4LDIwMjUtMTEtMDUsdHJ1ZSx0cnVlCjlQWkYsQl9DeXNfcG9zdFByb3Rlbml4LENZUywxMixTRyxBMUNRMixDMSwxLjk0MywyMDI2LTAzLTExLHRydWUsdHJ1ZQo5UFpZLEJfQ3lzX3Bvc3RQcm90ZW5peCxDWVMsMTIsU0csQTFBV1IsQzE3LDEuODQ1LDIwMjYtMDMtMTEsdHJ1ZSx0cnVlCjhTOEMsQ19DeXNfcG9zdEJvbHR6MixDWVMsMTIsU0csQTFINVUsQzQzLDEuODI1LDIwMjQtMDctMTAsdHJ1ZSxmYWxzZQo4VjM5LENfQ3lzX3Bvc3RCb2x0ejIsQ1lTLDEyLFNHLFk4TixDMzEsMS44MTQsMjAyNC0xMi0xOCx0cnVlLGZhbHNlCjhWM0EsQ19DeXNfcG9zdEJvbHR6MixDWVMsMTIsU0csWThOLEMzMSwxLjgxNCwyMDI0LTEyLTE4LHRydWUsZmFsc2UKOURNTSxDX0N5c19wb3N0Qm9sdHoyLENZUywxMixTRyxBMUFXUixDMTcsMS44MjksMjAyNS0wNS0yMSx0cnVlLGZhbHNlCjlFOUgsQ19DeXNfcG9zdEJvbHR6MixDWVMsMTIsU0csQTFCSDYsQzE2LDEuODA5LDIwMjUtMDItMjYsdHJ1ZSxmYWxzZQo5RTlJLENfQ3lzX3Bvc3RCb2x0ejIsQ1lTLDEyLFNHLEExQkg1LEMzMCwxLjgwOCwyMDI1LTAyLTI2LHRydWUsZmFsc2UKOUtQTSxDX0N5c19wb3N0Qm9sdHoyLENZUywxMixTRyxBMUw2QyxDMzAsMS43NjksMjAyNS0wMy0xMix0cnVlLGZhbHNlCjlLUE4sQ19DeXNfcG9zdEJvbHR6MixDWVMsMTIsU0csQTFMNkIsQzMwLDEuNzc1LDIwMjUtMDMtMTIsdHJ1ZSxmYWxzZQo="
}
for name, b in BLOBS.items():
    pathlib.Path(name).write_bytes(base64.b64decode(b))
    print(name, hashlib.sha256(pathlib.Path(name).read_bytes()).hexdigest()[:8])
print("expect kras_holdout.py d17a5e53")

## 3 · Install (clean Python 3.12 venv, chai_lab 0.6.1; ~5 min)

In [ ]:
!pip -q install uv
!uv venv --python 3.12 /content/cvenv -q
!uv pip install --python /content/cvenv/bin/python -q chai_lab==0.6.1 gemmi rdkit requests numpy
!/content/cvenv/bin/python -c "import torch, importlib.metadata as im; print('chai_lab', im.version('chai_lab'), 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!cd /content && /content/cvenv/bin/python kras_holdout.py prepare /content/work --manifest run_manifest.csv 2>&1 | tail -n 30

## 4 · Run Chai-1, SMILES arm only, in the background

In [ ]:
import subprocess, os, pathlib
os.chdir('/content')
pathlib.Path('/content/run.sh').write_text('''#!/bin/bash
set -o pipefail
cd /content
export MPLBACKEND=Agg CHAI_DOWNLOADS_DIR=/content/chai_downloads
/content/cvenv/bin/python kras_holdout.py predict-chai /content/work --arm smiles 2>&1 | grep -v Warning
echo "PREDICT_EXIT=$?"
DEST=/content/drive/MyDrive/kras_chai_smiles_2026-10-28
tar czf /content/chai_smiles_structures_2026-10-28.tgz -C /content work/out/chai work/manifest.json work/structures work/inputs/chai run_manifest.csv kras_holdout.py
cp /content/chai_smiles_structures_2026-10-28.tgz $DEST/
cp /content/work/manifest.json $DEST/
n=$(find /content/work/out/chai -name 'pred.model_idx_*.cif' | wc -l)
echo "structures=$n" | tee $DEST/RUN_COMPLETE.txt
echo ALL_DONE
''')
subprocess.run('pkill -f run.sh; true', shell=True)
subprocess.Popen('nohup bash /content/run.sh > /content/run.log 2>&1 &', shell=True)
print("started; re-run cell 5 for progress")

## 5 · Progress (re-run this cell)

In [ ]:
import subprocess, os
DEST='/content/drive/MyDrive/kras_chai_smiles_2026-10-28'
done = len(subprocess.run("find /content/work/out/chai -name 'pred.model_idx_*.cif' 2>/dev/null", shell=True, capture_output=True, text=True).stdout.split())
print(f"structures written: {done}/105")
print(subprocess.run('tail -n 15 /content/run.log', shell=True, capture_output=True, text=True).stdout)
print('--- Drive folder:', sorted(os.listdir(DEST)))

## 6 · After ALL_DONE: flush Drive and also download the zip

In [ ]:
from google.colab import drive, files
drive.flush_and_unmount()
files.download('/content/chai_smiles_structures_2026-10-28.tgz')